# SAVE (ICLR 2026) — Official Paper Reproduction Notebook (`sc-save` v0.2.0)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fdu-wangfeilab/sc-save/blob/main/notebooks/SAVE_ICLR2026_Reproduce_Colab.ipynb)

This self-contained notebook reproduces all five quantitative benchmark tables (**Tables 1–5**) from:
> **SAVE: A Generalizable Framework for Multi-Condition Single-Cell Generation with Gene Block Attention** (ICLR 2026)
> [OpenReview](https://openreview.net/forum?id=l7QEoK4uDP) · [arXiv:2604.16776](https://arxiv.org/abs/2604.16776)

### v0.2.0 Runtime & Execution Highlights
- **Google Colab Pro+**: **G4 (96 GB)**, **H100 (80 GB)**, or **A100 (40/80 GB)** with **High-RAM** enabled.
- **High-Throughput Execution**: Uses TF32, PyTorch SDPA, pure-PyTorch `CondOTPath` with fused `[2*B]` CFG sampling, zero-sync VAE/FM loops, adaptive GPU residency (`min(48 GB, 0.50 * free_vram)` FP32/FP16/pinned-CPU), precomputed frozen VAE latents, single-pass batched `generate_like` & BEC `transfer(..., input_cond="obs")`, and multi-worker GPU/CPU parallelism (`PREP_WORKERS = 4`, `GPU_WORKERS = "auto"`, `CPU_WORKERS = "auto"`).
- **Condition Encoding**: `COND_ENCODING = "legacy"` (default for 100% ICLR 2026 checkpoint/repro parity) or `"offset"` (selectable for collision-free cross-column condition IDs).
- **Top-to-bottom**: Run Cells **1 → 9** sequentially to reproduce all 5 tables (leave `RUN_ALL_PARALLEL = False` in the Fast Parallel Runner cell if running Cells 4–8 individually, or set `RUN_ALL_PARALLEL = True` and skip directly to Cell 9).
- **Selective by table**: Run Cells **1 → 3** once, run any subset of Tables (**Cells 4–8**) in any order, then run **Cell 9** to consolidate all completed results.
- **Optional Google Drive caching**: Toggle `USE_GOOGLE_DRIVE = True` in **Cell 2** to persist datasets, checkpoints, and outputs across sessions (`False` by default so external users require zero setup).


## Cell 1 — GPU, CUDA & PyTorch Check + Dependency Installation

Checks the attached GPU via `nvidia-smi` and verifies Colab's preinstalled CUDA-enabled PyTorch. Installs `sc-save` runtime dependencies **without** downgrading or replacing Colab's preinstalled CUDA `torch`.


In [ ]:
# === Cell 1: Hardware Check & Dependency Setup (Preserves Colab CUDA PyTorch) ===
import importlib.util
import shutil
import subprocess
import sys

print(f"Python: {sys.version.split()[0]}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("[warn] nvidia-smi not found (CPU runtime).")

import torch

pre_torch_version = torch.__version__
pre_cuda_version = torch.version.cuda
pre_cuda_avail = torch.cuda.is_available()
print(f"Preinstalled PyTorch: {pre_torch_version} | CUDA: {pre_cuda_version} | available={pre_cuda_avail}")
if pre_cuda_avail:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU[0]: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)")

# Install runtime dependencies matching pyproject.toml[all] without touching torch
REQUIRED_PACKAGES = [
    "anndata>=0.10.9",
    "scanpy>=1.10.0",
    "POT>=0.9.5",
    "einops>=0.8.0",
    "scib>=1.1.4",
    "igraph>=0.10.0",
    "leidenalg>=0.10.0",
    "gdown>=5.0.0",
    "pyyaml>=6.0",
    "requests>=2.31.0",
    "pandas>=2.2.0,<3.0",
    "scikit-learn>=1.5.0",
    "scipy>=1.13.0",
    "matplotlib>=3.8.0",
    "tqdm>=4.66.0",
    "pyparsing>=3.1.0",
]

print("\nInstalling sc-save runtime dependencies (preserving preinstalled torch)...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade-strategy",
        "only-if-needed",
        *REQUIRED_PACKAGES,
    ],
    check=True,
)

# Verify PyTorch CUDA stack stayed intact
import torch as _torch_after
assert _torch_after.__version__ == pre_torch_version, (
    f"PyTorch version unexpectedly changed: {pre_torch_version} -> {_torch_after.__version__}"
)
assert _torch_after.cuda.is_available() == pre_cuda_avail, (
    "PyTorch CUDA availability changed during pip install!"
)

for mod in ("anndata", "scanpy", "ot", "einops", "scib", "gdown"):
    assert importlib.util.find_spec(mod) is not None, f"Missing module after install: {mod}"
print(f"Environment ready. PyTorch={_torch_after.__version__}, CUDA available={_torch_after.cuda.is_available()}")

## Cell 2 — Clone / Sync `fdu-wangfeilab/sc-save` & Configure Parallelism + Cache

Clones `fdu-wangfeilab/sc-save` (or runs `git pull --ff-only` if already cloned on `BRANCH = "main"`), installs `sc-save` v0.2.0 in editable mode with `--no-deps`, sets parallel worker defaults (`PREP_WORKERS = 4`, `GPU_WORKERS = "auto"`, `CPU_WORKERS = "auto"`) and `COND_ENCODING` (`"legacy"` or `"offset"`), and optionally mounts Google Drive (`USE_GOOGLE_DRIVE = False` by default).


In [ ]:
# === Cell 2: Clone / Update Repository & Configure Workspace Paths + Parallelism ===
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys

import torch

# --- User Configuration ---
REPO_URL = "https://github.com/fdu-wangfeilab/sc-save.git"
BRANCH = "main"
USE_GOOGLE_DRIVE = False  # Set True to cache data, checkpoints, and results on Google Drive
DRIVE_WORK_DIR = "/content/drive/MyDrive/SAVE_ICLR2026"
LOCAL_WORK_DIR = "/content/save_iclr2026_work"

# Parallelism & condition encoding defaults
PREP_WORKERS = 4          # Concurrent raw downloads & OT Gene Block prep threads (or "auto")
GPU_WORKERS = "auto"      # "auto" (4-6 on >=75GB VRAM, 2 on >=36GB, 1 otherwise) or int
CPU_WORKERS = "auto"      # "auto" (min(8, cpu_count)) or int for scIB & distribution eval
COND_ENCODING = "legacy"  # "legacy" (paper parity) or "offset" (collision-free IDs)
# --------------------------

IN_COLAB = importlib.util.find_spec("google.colab") is not None or Path("/content").exists()
if Path.cwd().name == "sc-save" and (Path.cwd() / "examples" / "reproduce_iclr2026.py").exists():
    REPO_DIR = Path.cwd()
elif Path("/content").exists():
    REPO_DIR = Path("/content/sc-save")
else:
    REPO_DIR = Path.cwd() / "sc-save"

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} (branch={BRANCH}) -> {REPO_DIR}")
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Repository already present at {REPO_DIR}; syncing {BRANCH}...")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=False)

os.chdir(REPO_DIR)
for p in (str(REPO_DIR / "src"), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

# Editable install with --no-deps so Colab's preinstalled CUDA PyTorch stays untouched
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--no-build-isolation", "-e", str(REPO_DIR)],
    check=True,
)

import sc_save
print(f"Installed sc-save version: {sc_save.__version__}")

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = Path(DRIVE_WORK_DIR)
    except ImportError:
        print("[warn] google.colab not detected; using local workspace directory.")
        WORK_DIR = Path(LOCAL_WORK_DIR if Path("/content").exists() else "./save_iclr2026_work").resolve()
else:
    WORK_DIR = Path(LOCAL_WORK_DIR if Path("/content").exists() else "./save_iclr2026_work").resolve()

DATA_ROOT = WORK_DIR / "data"
OUTPUT_ROOT = WORK_DIR / "outputs" / "iclr2026"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / "results").mkdir(parents=True, exist_ok=True)

DEFAULT_DEVICE = 0 if torch.cuda.is_available() else -1
ENV_CONFIG = {
    "repo_dir": str(REPO_DIR),
    "branch": BRANCH,
    "use_google_drive": USE_GOOGLE_DRIVE,
    "work_dir": str(WORK_DIR),
    "data_root": str(DATA_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "device": DEFAULT_DEVICE,
    "prep_workers": PREP_WORKERS,
    "gpu_workers": GPU_WORKERS,
    "cpu_workers": CPU_WORKERS,
    "cond_encoding": COND_ENCODING,
}
Path("/tmp/save_colab_env.json").write_text(json.dumps(ENV_CONFIG, indent=2))


def _load_colab_env():
    cfg_path = Path("/tmp/save_colab_env.json")
    if cfg_path.exists():
        cfg = json.loads(cfg_path.read_text())
    else:
        repo = Path("/content/sc-save") if Path("/content/sc-save").exists() else Path.cwd()
        work = Path("/content/save_iclr2026_work") if Path("/content").exists() else Path("./save_iclr2026_work").resolve()
        cfg = {
            "repo_dir": str(repo),
            "data_root": str(work / "data"),
            "output_root": str(work / "outputs" / "iclr2026"),
            "device": 0 if torch.cuda.is_available() else -1,
            "prep_workers": 4,
            "gpu_workers": "auto",
            "cpu_workers": "auto",
            "cond_encoding": "legacy",
        }
    os.chdir(cfg["repo_dir"])
    for p in (str(Path(cfg["repo_dir"]) / "src"), cfg["repo_dir"]):
        if p not in sys.path:
            sys.path.insert(0, p)
    return cfg


def _stream_cmd(cmd: list[str]):
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    print("Running:", " ".join(str(x) for x in cmd))
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)


def _run_table_repro(
    table_name: str,
    datasets: str | None = None,
    device: int | None = None,
    seed: int = 1202,
    vae_epoch: int | None = None,
    fm_epoch: int | None = None,
    epoch: int | None = None,
    batch_size: int | None = None,
    subset_cells: int | None = None,
    max_target_batches: int = 0,
    table1_eval_split: str = "train",
    wd_mmd_repeats: int = 10,
    overwrite_ckpt: bool = False,
    gpu_workers: int | str | None = None,
    cpu_workers: int | str | None = None,
    prep_workers: int | str | None = None,
    cond_encoding: str | None = None,
    results_subdir: str | None = None,
):
    cfg = _load_colab_env()
    eff_device = cfg.get("device", 0) if device is None else device
    eff_gpu_workers = cfg.get("gpu_workers", "auto") if gpu_workers is None else gpu_workers
    eff_cpu_workers = cfg.get("cpu_workers", "auto") if cpu_workers is None else cpu_workers
    eff_prep_workers = cfg.get("prep_workers", 4) if prep_workers is None else prep_workers
    eff_cond_encoding = cfg.get("cond_encoding", "legacy") if cond_encoding is None else cond_encoding

    if results_subdir is None:
        results_dir = Path(cfg["output_root"]) / "results" / table_name
    elif results_subdir == "":
        results_dir = Path(cfg["output_root"]) / "results"
    else:
        results_dir = Path(cfg["output_root"]) / "results" / results_subdir
    results_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable,
        "-m",
        "sc_save.cli",
        "reproduce",
        "--tasks",
        table_name,
        "--data-root",
        cfg["data_root"],
        "--output-root",
        cfg["output_root"],
        "--results-dir",
        str(results_dir),
        "--device",
        str(eff_device),
        "--seed",
        str(seed),
        "--gpu-workers",
        str(eff_gpu_workers),
        "--cpu-workers",
        str(eff_cpu_workers),
        "--prep-workers",
        str(eff_prep_workers),
        "--cond-encoding",
        str(eff_cond_encoding),
    ]
    if datasets:
        cmd.extend(["--datasets", datasets])
    if vae_epoch is not None:
        cmd.extend(["--vae-epoch", str(vae_epoch)])
    if fm_epoch is not None:
        cmd.extend(["--fm-epoch", str(fm_epoch)])
    if epoch is not None:
        cmd.extend(["--epoch", str(epoch)])
    if batch_size is not None:
        cmd.extend(["--batch-size", str(batch_size)])
    if subset_cells is not None:
        cmd.extend(["--subset-cells", str(subset_cells)])
    if table_name in ("table4", "all") and max_target_batches > 0:
        cmd.extend(["--max-target-batches", str(max_target_batches)])
    if table_name in ("table1", "all"):
        cmd.extend(["--table1-eval-split", table1_eval_split])
    if table_name in ("table5", "all"):
        cmd.extend(["--wd-mmd-repeats", str(wd_mmd_repeats)])
    if overwrite_ckpt:
        cmd.append("--overwrite-ckpt")

    _stream_cmd(cmd)
    summary_path = results_dir / "iclr2026_summary.json"
    return json.loads(summary_path.read_text())


def _display_table_comparison(table_name: str, summary_payload: dict):
    import pandas as pd
    from IPython.display import display

    runs_root = summary_payload.get("runs", {})
    tables_to_show = [t for t in ("table1", "table2", "table3", "table4", "table5") if t in runs_root] if table_name == "all" else [table_name]
    rows = []
    for tname in tables_to_show:
        runs = runs_root.get(tname, {})
        if tname in ("table1", "table2"):
            for ds_name, entry in runs.items():
                obs, tgt, dlt = entry.get("observed", {}), entry.get("target", {}), entry.get("delta", {})
                for metric, tval in tgt.items():
                    rows.append({
                        "Table": tname.upper(),
                        "Dataset / Split": ds_name,
                        "Metric": metric,
                        "Reproduced": obs.get(metric),
                        "Paper Target": tval,
                        "Delta": dlt.get(metric),
                    })
        elif tname == "table3":
            entry = runs.get("lung_cancer", {})
            for split in ("seen", "unseen", "macro"):
                obs = entry.get("observed", {}).get(split, {})
                tgt = entry.get("target", {}).get(split, {})
                dlt = entry.get("delta", {}).get(split, {})
                for metric, tval in tgt.items():
                    rows.append({
                        "Table": "TABLE3",
                        "Dataset / Split": f"lung_cancer ({split})",
                        "Metric": metric,
                        "Reproduced": obs.get(metric),
                        "Paper Target": tval,
                        "Delta": dlt.get(metric),
                    })
        elif tname == "table4":
            for ds_name, entry in runs.items():
                obs, tgt, dlt = entry.get("observed", {}), entry.get("target", {}), entry.get("delta", {})
                for metric, tval in tgt.items():
                    rows.append({
                        "Table": "TABLE4",
                        "Dataset / Split": ds_name,
                        "Metric": metric,
                        "Reproduced": obs.get(metric),
                        "Paper Target": tval,
                        "Delta": dlt.get(metric),
                    })
        elif tname == "table5":
            entry = runs.get("pbmc_ifn", {})
            obs_all = entry.get("observed", {})
            tgt_all = entry.get("target", {})
            dlt_all = entry.get("delta", {})
            for ct, tgt in tgt_all.items():
                obs = obs_all.get(ct, {})
                dlt = dlt_all.get(ct, {})
                for metric in ("pcc", "r2", "wd", "mmd", "mse"):
                    if metric in tgt:
                        rows.append({
                            "Table": "TABLE5",
                            "Dataset / Split": f"pbmc_ifn ({ct})",
                            "Metric": metric,
                            "Reproduced": obs.get(metric),
                            "Paper Target": tgt[metric],
                            "Delta": dlt.get(metric),
                        })
    df = pd.DataFrame(rows)
    print(f"\n=== {table_name.upper()} — Reproduced vs. Paper Targets ===")
    display(df)
    return df


git_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print(
    f"Ready: repo={REPO_DIR} ({BRANCH}@{git_rev}) | sc-save={sc_save.__version__} | default_device={DEFAULT_DEVICE} | "
    f"PREP_WORKERS={PREP_WORKERS}, GPU_WORKERS={GPU_WORKERS}, CPU_WORKERS={CPU_WORKERS}, COND_ENCODING={COND_ENCODING}"
)
print(f"DATA_ROOT={DATA_ROOT}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")

## Cell 3 — Parallel Dataset Download & OT Gene Block Preprocessing

Uses `sc-save prepare-datasets --workers <PREP_WORKERS>` to download GenePT once, deduplicate shared raw `.h5ad` downloads across tables concurrently, apply paper-exact normalization (`log1p`-only for Table 2 dual-condition; `normalize_total(1e4)` + `log1p` for Tables 1, 3, 4, and 5), and run Optimal Transport balanced Gene Block construction (`ot.emd`, `numItermax=1_000_000`) in parallel.

- Set `PREPARE_DATASETS = ["all"]` for all 5 tables, or specify a subset such as `["table2"]` or `["heart", "pbmc_ifn"]`.
- Set `SUBSET_CELLS = None` for full ICLR 2026 reproduction (or e.g. `512` for a quick pipeline smoke test).


In [ ]:
# === Cell 3: Parallel Download & Preprocessing (Datasets + GenePT Embeddings) ===
import json
from pathlib import Path
import sys

cfg = _load_colab_env()

# --- Preparation Options ---
PREPARE_DATASETS = ["all"]   # e.g. ["all"], ["table2"], ["table1", "table2"], ["heart", "pbmc_ifn"]
PREP_WORKERS = cfg.get("prep_workers", 4)  # int >= 1 or "auto"
SUBSET_CELLS = None          # None for full datasets; set integer (e.g. 512) for fast smoke test
OT_NUM_ITER_MAX = 1_000_000
FORCE_PREPARE = False
SKIP_DOWNLOAD = False
# ---------------------------

cmd = [
    sys.executable,
    "-m",
    "sc_save.cli",
    "prepare-datasets",
    "--datasets",
    *PREPARE_DATASETS,
    "--data-root",
    cfg["data_root"],
    "--workers",
    str(PREP_WORKERS),
    "--ot-num-iter-max",
    str(OT_NUM_ITER_MAX),
]
if SUBSET_CELLS is not None:
    cmd.extend(["--subset-cells", str(SUBSET_CELLS)])
if FORCE_PREPARE:
    cmd.append("--force")
if SKIP_DOWNLOAD:
    cmd.append("--skip-download")

_stream_cmd(cmd)

manifest_path = Path(cfg["data_root"]) / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(f"\nPrepared {len(manifest.get('datasets', {}))} datasets in {manifest_path}:")
    for name, info in manifest.get("datasets", {}).items():
        print(f"  - {name:16s} | shape=({info['n_obs']}, {info['n_vars']}) | GBS={info['gene_block_size']} | path={info['primary_path']}")

## Fast Parallel All-Tables Runner (Optional — G4 / H100 / A100 Full GPU Saturation)

Runs **all 5 paper tables (`--tasks all`)** concurrently with `--gpu-workers auto --cpu-workers auto --prep-workers 4` in a single command, writing `<OUTPUT_ROOT>/results/iclr2026_summary.json` and per-task logs under `<OUTPUT_ROOT>/logs/`.

- Leave `RUN_ALL_PARALLEL = False` if you prefer running **Cells 4–8** selectively or table-by-table below.
- Set `RUN_ALL_PARALLEL = True` (or run directly) to saturate G4 (96 GB) / H100 (80 GB) / A100 GPUs across all benchmarks, then jump straight to **Cell 9**!


In [ ]:
# === Fast Parallel All-Tables Runner (Optional One-Click Full Saturation) ===
RUN_ALL_PARALLEL = False    # Set True to run Tables 1-5 concurrently, then jump to Cell 9
GPU_WORKERS_ALL = "auto"    # "auto" (4-6 on G4 96GB / H100 80GB, 2 on A100 40GB) or int
CPU_WORKERS_ALL = "auto"    # "auto" (min(8, cpu_count)) or int
PREP_WORKERS_ALL = 4
VAE_EPOCH_ALL = 200
FM_EPOCH_ALL = 200
EPOCH_ALL = 200
BATCH_SIZE_ALL = 1024
SUBSET_CELLS_ALL = None     # Optional integer cap for smoke testing
OVERWRITE_CKPT_ALL = False

if RUN_ALL_PARALLEL:
    all_summary = _run_table_repro(
        table_name="all",
        seed=1202,
        vae_epoch=VAE_EPOCH_ALL,
        fm_epoch=FM_EPOCH_ALL,
        epoch=EPOCH_ALL,
        batch_size=BATCH_SIZE_ALL,
        subset_cells=SUBSET_CELLS_ALL,
        overwrite_ckpt=OVERWRITE_CKPT_ALL,
        gpu_workers=GPU_WORKERS_ALL,
        cpu_workers=CPU_WORKERS_ALL,
        prep_workers=PREP_WORKERS_ALL,
        results_subdir="",
    )
    _display_table_comparison("all", all_summary)
else:
    print("RUN_ALL_PARALLEL=False; skipping batch runner. Run Cells 4-8 individually or set RUN_ALL_PARALLEL=True.")

## Cell 4 — Table 2: Dual-Condition Generation (`Heart`, `PBMC`, `Lung Atlas`)

Trains and evaluates `SAVE-B` (`gene_block_size=3200`, `cond_cols=["batch", "cell_type"]`, `vae_epoch=200`, `fm_epoch=200`, `sample_mask_prob=0.4`, `guidance_weights=2.0`, `sample_steps=20`) concurrently across `heart`, `pbmc`, and `lung_atlas` using `GPU_WORKERS = "auto"`, `CPU_WORKERS = "auto"`, and single-pass batched `generate_like`.


In [ ]:
# === Cell 4: Run Table 2 — Dual-Condition Generation (Heart, PBMC, Lung Atlas) ===
TABLE2_DATASETS = None      # None runs all (heart,pbmc,lung_atlas); or e.g. "heart,pbmc"
VAE_EPOCH_T2 = 200
FM_EPOCH_T2 = 200
BATCH_SIZE_T2 = 1024
SUBSET_CELLS_T2 = None      # Optional integer cap for smoke testing
OVERWRITE_CKPT_T2 = False

t2_summary = _run_table_repro(
    table_name="table2",
    datasets=TABLE2_DATASETS,
    seed=1202,
    vae_epoch=VAE_EPOCH_T2,
    fm_epoch=FM_EPOCH_T2,
    batch_size=BATCH_SIZE_T2,
    subset_cells=SUBSET_CELLS_T2,
    overwrite_ckpt=OVERWRITE_CKPT_T2,
)
_display_table_comparison("table2", t2_summary)

## Cell 5 — Table 1: Single-Condition Generation (`PBMC3K`, `Dentate gyrus`, `Tabula Muris`)

Trains and evaluates `SAVE-B` (`gene_block_size=3200`, `cond_cols=["cell_type"]`, `vae_epoch=200`, `fm_epoch=200`, `sample_mask_prob=0.4`) concurrently across `pbmc3k`, `dentategyrus`, and `tabula_muris`.


In [ ]:
# === Cell 5: Run Table 1 — Single-Condition Generation (PBMC3K, Dentate gyrus, Tabula Muris) ===
TABLE1_DATASETS = None      # None runs all (pbmc3k,dentategyrus,tabula_muris)
TABLE1_EVAL_SPLIT = "train" # "train" (paper default) or "test"
VAE_EPOCH_T1 = 200
FM_EPOCH_T1 = 200
BATCH_SIZE_T1 = 1024
SUBSET_CELLS_T1 = None
OVERWRITE_CKPT_T1 = False

t1_summary = _run_table_repro(
    table_name="table1",
    datasets=TABLE1_DATASETS,
    seed=1202,
    vae_epoch=VAE_EPOCH_T1,
    fm_epoch=FM_EPOCH_T1,
    batch_size=BATCH_SIZE_T1,
    subset_cells=SUBSET_CELLS_T1,
    table1_eval_split=TABLE1_EVAL_SPLIT,
    overwrite_ckpt=OVERWRITE_CKPT_T1,
)
_display_table_comparison("table1", t1_summary)

## Cell 6 — Table 3: Multi-Condition Generation (`Lung Cancer` Seen & Unseen 13, 24)

Trains `SAVE` (`gene_block_size=400`, `HVG=2000`, `cond_cols=["assay", "development_stage", "disease", "uicc_stage", "cell_type"]`, `sample_mask_prob=0.0`) on the 25 Seen conditions of Salcher et al. Lung Cancer and evaluates both **Seen** and held-out **Unseen** combinations (conditions **13** and **24**) plus the 27-condition macro average.


In [ ]:
# === Cell 6: Run Table 3 — Multi-Condition Generation (Lung Cancer Seen & Unseen 13, 24) ===
VAE_EPOCH_T3 = 200
FM_EPOCH_T3 = 200
BATCH_SIZE_T3 = 1024
SUBSET_CELLS_T3 = None
OVERWRITE_CKPT_T3 = False

t3_summary = _run_table_repro(
    table_name="table3",
    datasets="lung_cancer",
    seed=1202,
    vae_epoch=VAE_EPOCH_T3,
    fm_epoch=FM_EPOCH_T3,
    batch_size=BATCH_SIZE_T3,
    subset_cells=SUBSET_CELLS_T3,
    overwrite_ckpt=OVERWRITE_CKPT_T3,
)
_display_table_comparison("table3", t3_summary)

## Cell 7 — Table 4: Batch Effect Correction (`Lung Atlas`, `Heart`, `PBMC`) + scIB Evaluation

Runs `SAVE-B-perturb` `LDM_raw` conditional batch transfer (`gene_block_size=400`, `HVG=2000`, `sample_mask_prob=0.0`, `col_msk_threshold=0.6`) in a single batched pass per target batch and evaluates `scib` metrics in parallel across CPU workers (`avg_bio`, `avg_batch`, `scib_score = 0.6 * avg_bio + 0.4 * avg_batch`).


In [ ]:
# === Cell 7: Run Table 4 — Batch Effect Correction (Lung Atlas, Heart, PBMC) + scIB ===
TABLE4_DATASETS = None      # None runs all (lung_atlas_bec,heart_bec,pbmc_bec)
EPOCH_T4 = 200
BATCH_SIZE_T4 = 1024
MAX_TARGET_BATCHES_T4 = 0   # 0 evaluates all target batches
SUBSET_CELLS_T4 = None
OVERWRITE_CKPT_T4 = False

t4_summary = _run_table_repro(
    table_name="table4",
    datasets=TABLE4_DATASETS,
    seed=1202,
    epoch=EPOCH_T4,
    batch_size=BATCH_SIZE_T4,
    subset_cells=SUBSET_CELLS_T4,
    max_target_batches=MAX_TARGET_BATCHES_T4,
    overwrite_ckpt=OVERWRITE_CKPT_T4,
)
_display_table_comparison("table4", t4_summary)

## Cell 8 — Table 5: Leave-One-Out Perturbation Prediction (`PBMC-IFN`, all 7 cell types)

Runs Leave-One-Out counterfactual IFN-$\beta$ perturbation prediction across all 7 PBMC cell types (`CD8T`, `CD14+Mono`, `FCGR3A+Mono`, `B`, `CD4T`, `Dendritic`, `NK`) using `SAVE-B-perturb` (`gene_block_size=400`, `train_with_ctrl=False`, `sample_mask_prob=0.0`, `col_msk_threshold=0.6`) with `GPU_WORKERS = "auto"` (saturating up to 7 concurrent LOO folds on G4/H100 GPUs) and evaluates `PCC`, `R2`, `MSE`, `WD`, and `MMD`.


In [ ]:
# === Cell 8: Run Table 5 — Leave-One-Out Perturbation Prediction (PBMC-IFN) ===
EPOCH_T5 = 200
BATCH_SIZE_T5 = 1024
WD_MMD_REPEATS_T5 = 10
SUBSET_CELLS_T5 = None

t5_summary = _run_table_repro(
    table_name="table5",
    datasets="pbmc_ifn",
    seed=1202,
    epoch=EPOCH_T5,
    batch_size=BATCH_SIZE_T5,
    subset_cells=SUBSET_CELLS_T5,
    wd_mmd_repeats=WD_MMD_REPEATS_T5,
)
_display_table_comparison("table5", t5_summary)

## Cell 9 — Consolidated Paper vs. Reproduced Comparison & Export (`iclr2026_colab_repro.json`)

Aggregates every completed table from `<OUTPUT_ROOT>/results/table{1..5}/iclr2026_summary.json` and `<OUTPUT_ROOT>/results/iclr2026_summary.json`, prints a unified Paper vs. Reproduced comparison table across Tables 1–5, saves `iclr2026_colab_repro.json`, and downloads it in Google Colab.


In [ ]:
# === Cell 9: Consolidated Comparison Table & Export iclr2026_colab_repro.json ===
from datetime import datetime, timezone
import importlib.util
import json
import os
from pathlib import Path
import platform
import shutil
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

# Self-contained environment resolution (works even after runtime reconnect)
if "_load_colab_env" in globals():
    cfg = _load_colab_env()
else:
    cfg_file = Path("/tmp/save_colab_env.json")
    if cfg_file.exists():
        cfg = json.loads(cfg_file.read_text())
    else:
        _repo = Path("/content/sc-save") if Path("/content/sc-save").exists() else Path.cwd()
        _work = Path("/content/save_iclr2026_work") if Path("/content").exists() else Path("./save_iclr2026_work").resolve()
        cfg = {
            "repo_dir": str(_repo),
            "data_root": str(_work / "data"),
            "output_root": str(_work / "outputs" / "iclr2026"),
            "prep_workers": 4,
            "gpu_workers": "auto",
            "cpu_workers": "auto",
            "cond_encoding": "legacy",
        }
    os.chdir(cfg["repo_dir"])
    for p in (str(Path(cfg["repo_dir"]) / "src"), cfg["repo_dir"]):
        if p not in sys.path:
            sys.path.insert(0, p)

repo_dir = Path(cfg["repo_dir"])
output_root = Path(cfg["output_root"])
results_root = output_root / "results"
AUTO_DOWNLOAD = True

import sc_save
from sc_save.workflows.reproduce import PAPER_TARGETS, VALID_TABLES, render_summary_markdown

# Merge per-table summaries and any top-level all-tables summary
merged_runs = {}
fallback_summary = results_root / "iclr2026_summary.json"
if fallback_summary.exists():
    fb_data = json.loads(fallback_summary.read_text())
    for tname, truns in fb_data.get("runs", {}).items():
        merged_runs.setdefault(tname, {}).update(truns)

for tname in VALID_TABLES:
    t_summary = results_root / tname / "iclr2026_summary.json"
    if t_summary.exists():
        t_data = json.loads(t_summary.read_text())
        for sub_t, sub_runs in t_data.get("runs", {}).items():
            merged_runs.setdefault(sub_t, {}).update(sub_runs)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
gpu_vram_gb = (
    round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    if torch.cuda.is_available()
    else 0.0
)
try:
    git_branch = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
    git_commit = subprocess.check_output(["git", "-C", str(repo_dir), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception:
    git_branch, git_commit = "unknown", "unknown"

consolidated_payload = {
    "metadata": {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "dry_run": False,
        "sc_save_version": sc_save.__version__,
        "cond_encoding": cfg.get("cond_encoding", "legacy"),
        "data_root": cfg["data_root"],
        "output_root": cfg["output_root"],
        "git_branch": git_branch,
        "git_commit": git_commit,
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "gpu_name": gpu_name,
        "gpu_vram_gb": gpu_vram_gb,
        "prep_workers": cfg.get("prep_workers", 4),
        "gpu_workers": cfg.get("gpu_workers", "auto"),
        "cpu_workers": cfg.get("cpu_workers", "auto"),
        "tables_completed": [t for t in VALID_TABLES if t in merged_runs],
    },
    "paper_targets": PAPER_TARGETS,
    "runs": merged_runs,
}

# Build unified comparison DataFrame across Tables 1-5
rows = []
for tname in VALID_TABLES:
    truns = merged_runs.get(tname, {})
    if tname in ("table1", "table2"):
        for ds_name, entry in truns.items():
            obs, tgt, dlt = entry.get("observed", {}), entry.get("target", {}), entry.get("delta", {})
            for metric in ("wd", "mmd", "mse", "pcc", "r2"):
                if metric in tgt:
                    rows.append({
                        "Table": tname.upper(),
                        "Benchmark": ds_name,
                        "Metric": metric,
                        "Reproduced": obs.get(metric),
                        "Paper": tgt[metric],
                        "Delta": dlt.get(metric),
                    })
    elif tname == "table3" and "lung_cancer" in truns:
        entry = truns["lung_cancer"]
        for split in ("seen", "unseen", "macro"):
            obs = entry.get("observed", {}).get(split, {})
            tgt = entry.get("target", {}).get(split, {})
            dlt = entry.get("delta", {}).get(split, {})
            for metric, tval in tgt.items():
                rows.append({
                    "Table": "TABLE3",
                    "Benchmark": f"lung_cancer ({split})",
                    "Metric": metric,
                    "Reproduced": obs.get(metric),
                    "Paper": tval,
                    "Delta": dlt.get(metric),
                })
    elif tname == "table4":
        for ds_name, entry in truns.items():
            obs, tgt, dlt = entry.get("observed", {}), entry.get("target", {}), entry.get("delta", {})
            for metric in ("avg_bio", "avg_batch", "scib_score", "NMI_cluster/label", "ARI_cluster/label", "ASW_label", "ASW_label/batch", "PCR_batch", "graph_conn"):
                if metric in tgt:
                    rows.append({
                        "Table": "TABLE4",
                        "Benchmark": ds_name,
                        "Metric": metric,
                        "Reproduced": obs.get(metric),
                        "Paper": tgt[metric],
                        "Delta": dlt.get(metric),
                    })
    elif tname == "table5" and "pbmc_ifn" in truns:
        entry = truns["pbmc_ifn"]
        obs_all = entry.get("observed", {})
        tgt_all = entry.get("target", {})
        dlt_all = entry.get("delta", {})
        for ct, tgt in tgt_all.items():
            obs = obs_all.get(ct, {})
            dlt = dlt_all.get(ct, {})
            for metric in ("pcc", "r2", "wd", "mmd", "mse"):
                if metric in tgt:
                    rows.append({
                        "Table": "TABLE5",
                        "Benchmark": f"pbmc_ifn ({ct})",
                        "Metric": metric,
                        "Reproduced": obs.get(metric),
                        "Paper": tgt[metric],
                        "Delta": dlt.get(metric),
                    })

comparison_df = pd.DataFrame(rows)
print(f"Completed tables: {consolidated_payload['metadata']['tables_completed']} | Hardware: {gpu_name} ({gpu_vram_gb} GB)")
if len(comparison_df) > 0:
    display(comparison_df)
else:
    print("[info] No completed table outputs found yet. Run the Fast Parallel Runner or Cells 4-8 first.")

output_root.mkdir(parents=True, exist_ok=True)
out_json_work = output_root / "iclr2026_colab_repro.json"
out_md_work = output_root / "iclr2026_colab_repro.md"
out_json_work.write_text(json.dumps(consolidated_payload, indent=2))
out_md_work.write_text(render_summary_markdown(consolidated_payload))

repo_results_dir = repo_dir / "results"
repo_results_dir.mkdir(parents=True, exist_ok=True)
out_json_repo = repo_results_dir / "iclr2026_colab_repro.json"
shutil.copy2(out_json_work, out_json_repo)

print(f"Saved JSON to: {out_json_work}")
print(f"Copied JSON to: {out_json_repo}")

if AUTO_DOWNLOAD and importlib.util.find_spec("google.colab") is not None:
    try:
        from google.colab import files
        files.download(str(out_json_work))
    except Exception as exc:
        print(f"[info] Colab auto-download skipped ({exc}); file remains saved at {out_json_work}")